In [ ]:
# ============================================================
# Cell 0 — Notebook Overview: Classifier Selection
# File: 06.5_Classifier_Selection.ipynb
# ============================================================

# PURPOSE:
# This notebook performs classifier selection to determine which
# scikit-learn classification algorithm is most effective for
# distinguishing AI-generated and real images using the previously
# extracted 26-dimensional Digital Image Processing (DIP) feature vectors.
#
# Multiple classifiers are evaluated using the same normalized
# training data so that classifier performance can be compared
# fairly and consistently. The strongest classifier candidates
# identified here will guide subsequent final training,
# hyperparameter tuning, and independent test evaluation.

# INPUTS:
# - Normalized classifier-input datasets generated from:
#   06_Normalize_and_Prepare_Classifier_Input.ipynb
#
# Expected inputs include:
#   - X_train (normalized feature matrix, shape: 8400 x 26)
#   - y_train (class labels: 'ai' or 'real')
#   - (optional) X_validation
#   - (optional) y_validation
#
# Depending on implementation, these may be loaded as:
#   - separate CSV files for X_train and y_train
#   - separate CSV files for training and validation datasets
#   - serialized Python objects saved from the prior notebook

# NOTE:
# All features have already been normalized using a scaler fit only
# on the training dataset in the previous notebook. No additional
# normalization should be applied here.

# ASSUMPTIONS:
# - Dataset collection, preprocessing, and splitting are complete
# - DIP feature extraction is complete
# - Final classifier-input matrices have already been prepared
# - Training, validation, and test subsets remain strictly separated
# - Features are numerical and free of missing values
# - Training labels are correctly aligned with the feature rows
# - The 26-feature DIP vector is the common input to every classifier
# - scikit-learn is available in the current Python environment

# PROCESS OVERVIEW:
# This notebook uses scikit-learn to compare a broad set of
# classification algorithms on the same normalized DIP feature set.
# The workflow consists of:
#
# - loading and validating the training dataset
# - defining a set of candidate classifiers
# - evaluating baseline classifier performance using stratified cross-validation
# - ranking classifiers using multiple performance metrics
# - applying light hyperparameter tuning to top-performing classifiers
# - saving comparison results for reporting
# - recommending classifier(s) for subsequent training and evaluation

# OUTPUTS:
# - Ranked classifier comparison table (baseline results)
# - Identification of top-performing classifier candidates
# - Lightly tuned classifier results for top models
# - Saved CSV files:
#     - classifier_comparison_baseline.csv
#     - classifier_comparison_tuned.csv
# - Summary of recommended classifier(s) for subsequent notebooks
#
# These results support classifier choice for later training
# notebooks and provide evidence for comparing classifier
# performance on the same DIP feature set.

# KEY DESIGN CHOICE:
# This notebook performs classifier selection only.
# It is intended to compare different classifier types
# (for example, Logistic Regression, SVM, Random Forest,
# Extra Trees, Gradient Boosting, and MLPClassifier)
# using the same input features.
#
# It does NOT perform:
# - final full hyperparameter optimization
# - final independent test-set evaluation
# - replacement of the independent test procedure
#
# Those steps belong in later notebooks.

# TERMINOLOGY NOTE:
# In this notebook, the primary comparison is across classifiers
# (algorithm types), not across fully optimized final models.
# Each classifier evaluated here produces a trained model instance,
# but the main purpose of the notebook is classifier selection.

# IMPORTANT NOTES:
# - Input features are already scaled from Notebook 06
# - All classifiers must use the same training data for fair comparison
# - Validation and test datasets should remain untouched unless
#   explicitly used later for follow-up evaluation
# - Metrics such as Accuracy, Precision, Recall, F1 Score,
#   and ROC-AUC are especially relevant for classifier selection
# - Tree-based classifiers do not require scaling, but scaled inputs
#   are retained here for consistency across all classifiers

# ============================================================
# CELL-BY-CELL STRUCTURE
# ============================================================

# Cell 1:
# - Import required libraries
#   (pandas, numpy, sklearn classifiers, sklearn metrics,
#    sklearn model_selection tools, os, warnings, etc.)

# Cell 2:
# - Define file paths or input sources
# - Load normalized training features and labels
# - Optionally load validation features and labels
# - Display dataset shapes and column names

# Cell 3:
# - Perform sanity checks
#   - missing values
#   - duplicate rows (optional)
#   - class distribution
#   - expected feature count (26)

# Cell 4:
# - Define a dictionary of candidate classifiers
# - Establish baseline settings for each classifier
# - Prepare scoring metrics for comparison

# Cell 5:
# - Run classifier comparison using stratified cross-validation
# - Collect performance metrics for each classifier

# Cell 6:
# - Assemble results into a ranked comparison table
# - Sort classifiers by selected metric
# - Display the strongest baseline classifiers

# Cell 7:
# - Select the top classifier candidates
# - Apply light hyperparameter tuning to a small number
#   of leading classifiers using GridSearchCV
# - Re-evaluate tuned versions

# Cell 8:
# - Save classifier comparison results to CSV
# - Generate compact reporting tables for top classifiers

# Cell 9:
# - Summarize findings
#   - best baseline classifier
#   - best tuned classifier
#   - key observations

# ============================================================
# END OF NOTEBOOK OVERVIEW
# ============================================================


In [ ]:
# ============================================================
# Cell 1 — Import Libraries (scikit-learn Classifier Selection)
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

# Scikit-learn: model selection & evaluation
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
# ============================================================
# Cell 2 — Load Normalized Training and Validation Data
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# Update base directory if needed
# ------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/DIP_Project"

TRAIN_PATH = os.path.join(BASE_DIR, "train_feature_vectors_normalized.csv")
VAL_PATH   = os.path.join(BASE_DIR, "validation_feature_vectors_normalized.csv")

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------
required_files = [TRAIN_PATH, VAL_PATH]
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    raise FileNotFoundError(
        "Missing required input files:\n" + "\n".join(missing_files)
    )

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

print("Datasets loaded successfully.\n")
print("Train shape      :", train_df.shape)
print("Validation shape :", val_df.shape)

# ------------------------------------------------------------
# Separate features and labels
# ------------------------------------------------------------
TARGET_COL = "class_label"

NON_FEATURE_COLS = ["filename", "source_dataset", "subset", TARGET_COL]

feature_cols = [col for col in train_df.columns if col not in NON_FEATURE_COLS]

X_train_df = train_df[feature_cols].copy()
y_train    = train_df[TARGET_COL].copy()

X_val_df   = val_df[feature_cols].copy()
y_val      = val_df[TARGET_COL].copy()

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------
print("\nFeature matrix shapes:")
print("X_train_df :", X_train_df.shape)
print("X_val_df   :", X_val_df.shape)

print("\nLabel shapes:")
print("y_train    :", y_train.shape)
print("y_val      :", y_val.shape)

print("\nLabel distribution (train):")
print(y_train.value_counts())

print("\nNumber of selected features:", len(feature_cols))
print("First 5 feature columns:", feature_cols[:5])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Datasets loaded successfully.

Train shape      : (8400, 30)
Validation shape : (1800, 30)

Feature matrix shapes:
X_train_df : (8400, 26)
X_val_df   : (1800, 26)

Label shapes:
y_train    : (8400,)
y_val      : (1800,)

Label distribution (train):
class_label
ai      4200
real    4200
Name: count, dtype: int64

Number of selected features: 26
First 5 feature columns: ['Mean Gradient', 'Std Gradient', 'Max Gradient', 'Gradient Entropy', 'Edge Density']


In [ ]:
# ============================================================
# Cell 3 — Sanity Checks
# ============================================================

print("Running sanity checks...\n")

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------
train_missing = X_train_df.isnull().sum().sum()
val_missing   = X_val_df.isnull().sum().sum()
y_train_missing = y_train.isnull().sum()
y_val_missing   = y_val.isnull().sum()

print("Missing values check:")
print("X_train_df :", train_missing)
print("X_val_df   :", val_missing)
print("y_train    :", y_train_missing)
print("y_val      :", y_val_missing)

assert train_missing == 0, "Missing values found in X_train_df"
assert val_missing == 0, "Missing values found in X_val_df"
assert y_train_missing == 0, "Missing values found in y_train"
assert y_val_missing == 0, "Missing values found in y_val"

# ------------------------------------------------------------
# Duplicate rows
# ------------------------------------------------------------
train_duplicates = X_train_df.duplicated().sum()
val_duplicates   = X_val_df.duplicated().sum()

print("\nDuplicate feature rows:")
print("X_train_df :", train_duplicates)
print("X_val_df   :", val_duplicates)

# ------------------------------------------------------------
# Feature count
# ------------------------------------------------------------
expected_feature_count = 26

print("\nFeature count check:")
print("Observed feature count:", X_train_df.shape[1])
print("Expected feature count:", expected_feature_count)

assert X_train_df.shape[1] == expected_feature_count, \
    f"Expected {expected_feature_count} features, found {X_train_df.shape[1]}"

assert X_val_df.shape[1] == expected_feature_count, \
    f"Expected {expected_feature_count} features, found {X_val_df.shape[1]}"

# ------------------------------------------------------------
# Column consistency
# ------------------------------------------------------------
same_columns = list(X_train_df.columns) == list(X_val_df.columns)

print("\nColumn consistency check:")
print("Train/validation feature columns match:", same_columns)

assert same_columns, "Feature columns do not match between training and validation sets"

# ------------------------------------------------------------
# Label consistency
# ------------------------------------------------------------
train_labels = sorted(y_train.unique())
val_labels   = sorted(y_val.unique())

print("\nLabel set check:")
print("Training labels   :", train_labels)
print("Validation labels :", val_labels)

assert train_labels == val_labels, "Training and validation label sets do not match"

# ------------------------------------------------------------
# Basic normalization spot-check
# ------------------------------------------------------------
train_means = X_train_df.mean()
train_stds  = X_train_df.std()

print("\nNormalization spot-check (first 5 features):")
print("Means:")
print(train_means.head())

print("\nStds:")
print(train_stds.head())

print("\nAll sanity checks passed.")


Running sanity checks...

Missing values check:
X_train_df : 0
X_val_df   : 0
y_train    : 0
y_val      : 0

Duplicate feature rows:
X_train_df : 0
X_val_df   : 0

Feature count check:
Observed feature count: 26
Expected feature count: 26

Column consistency check:
Train/validation feature columns match: True

Label set check:
Training labels   : ['ai', 'real']
Validation labels : ['ai', 'real']

Normalization spot-check (first 5 features):
Means:
Mean Gradient      -2.239478e-16
Std Gradient       -2.916186e-16
Max Gradient        3.049413e-16
Gradient Entropy    3.793791e-16
Edge Density       -2.453064e-17
dtype: float64

Stds:
Mean Gradient       1.00006
Std Gradient        1.00006
Max Gradient        1.00006
Gradient Entropy    1.00006
Edge Density        1.00006
dtype: float64

All sanity checks passed.


In [ ]:
# ============================================================
# Cell 4 — Define Candidate Classifiers and Scoring Metrics
# ============================================================

# Cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scoring metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc"
}

# Candidate classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=42
    ),

    "RBF SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(random_state=42),

    "AdaBoost": AdaBoostClassifier(random_state=42),

    "MLPClassifier": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=500,
        random_state=42
    ),

    "Gaussian Naive Bayes": GaussianNB(),

    "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),

    "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis()
}

print("Cross-validation strategy:")
print(cv_strategy)

print("\nScoring metrics:")
for metric_name in scoring:
    print("-", metric_name)

print("\nCandidate classifiers:")
for i, clf_name in enumerate(classifiers.keys(), start=1):
    print(f"{i:2d}. {clf_name}")

print("\nTotal classifiers defined:", len(classifiers))


Cross-validation strategy:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

Scoring metrics:
- accuracy
- precision
- recall
- f1
- roc_auc

Candidate classifiers:
 1. Logistic Regression
 2. Linear SVM
 3. RBF SVM
 4. KNN
 5. Decision Tree
 6. Random Forest
 7. Extra Trees
 8. Gradient Boosting
 9. AdaBoost
10. MLPClassifier
11. Gaussian Naive Bayes
12. Linear Discriminant Analysis
13. Quadratic Discriminant Analysis

Total classifiers defined: 13


In [ ]:
# ============================================================
# Cell 5 — Run Cross-Validated Classifier Comparison
# ============================================================

results = []

print("Running cross-validated classifier comparison...\n")

for clf_name, clf in classifiers.items():
    print(f"Evaluating: {clf_name}")

    cv_scores = cross_validate(
        estimator=clf,
        X=X_train_df,
        y=y_train,
        cv=cv_strategy,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    result_row = {
        "classifier": clf_name,
        "accuracy":  np.mean(cv_scores["test_accuracy"]),
        "precision": np.mean(cv_scores["test_precision"]),
        "recall":    np.mean(cv_scores["test_recall"]),
        "f1_score":  np.mean(cv_scores["test_f1"]),
        "roc_auc":   np.mean(cv_scores["test_roc_auc"])
    }

    results.append(result_row)

print("\nCross-validation complete.")
print(f"Total classifiers evaluated: {len(results)}")


Running cross-validated classifier comparison...

Evaluating: Logistic Regression
Evaluating: Linear SVM
Evaluating: RBF SVM
Evaluating: KNN
Evaluating: Decision Tree
Evaluating: Random Forest
Evaluating: Extra Trees
Evaluating: Gradient Boosting
Evaluating: AdaBoost
Evaluating: MLPClassifier
Evaluating: Gaussian Naive Bayes
Evaluating: Linear Discriminant Analysis
Evaluating: Quadratic Discriminant Analysis

Cross-validation complete.
Total classifiers evaluated: 13


In [ ]:
# ============================================================
# Cell 6 — Assemble and Rank Classifier Comparison Results
# ============================================================

results_df = pd.DataFrame(results)

# Sort primarily by ROC-AUC, then by F1 score, then by Accuracy
results_df = results_df.sort_values(
    by=["roc_auc", "f1_score", "accuracy"],
    ascending=False
).reset_index(drop=True)

# Add ranking column
results_df.insert(0, "rank", range(1, len(results_df) + 1))

# Optional formatting for cleaner display
results_display_df = results_df.copy()
metric_cols = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
results_display_df[metric_cols] = results_display_df[metric_cols].round(6)

print("Ranked classifier comparison results:\n")
print(results_display_df)

print("\nTop 5 classifiers:")
print(results_display_df.head(5))


Ranked classifier comparison results:

    rank                       classifier  accuracy  precision    recall  f1_score   roc_auc
0      1                    MLPClassifier  0.864643   0.865250  0.864643  0.864588  0.935720
1      2                          RBF SVM  0.846429   0.846478  0.846429  0.846423  0.925766
2      3                    Random Forest  0.844286   0.844293  0.844286  0.844285  0.921758
3      4                      Extra Trees  0.839286   0.839438  0.839286  0.839268  0.919930
4      5                Gradient Boosting  0.809881   0.810118  0.809881  0.809843  0.897675
5      6                              KNN  0.813333   0.816636  0.813333  0.812837  0.884283
6      7  Quadratic Discriminant Analysis  0.815595   0.816231  0.815595  0.815501  0.878779
7      8                       Linear SVM  0.790833   0.791051  0.790833  0.790794  0.864741
8      9              Logistic Regression  0.788214   0.788364  0.788214  0.788186  0.864007
9     10                       

In [ ]:
# ============================================================
# Cell 7 — Light Hyperparameter Tuning (Top Classifiers)
# ============================================================

from sklearn.model_selection import GridSearchCV

print("Starting light hyperparameter tuning...\n")

tuned_results = []

# ------------------------------------------------------------
# 1. MLPClassifier tuning
# ------------------------------------------------------------
mlp_param_grid = {
    "hidden_layer_sizes": [(64,32), (128,64), (64,64)],
    "alpha": [0.0001, 0.001],
}

mlp_grid = GridSearchCV(
    MLPClassifier(max_iter=500, random_state=42),
    mlp_param_grid,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=-1
)

mlp_grid.fit(X_train_df, y_train)

tuned_results.append({
    "classifier": "MLP (tuned)",
    "best_params": mlp_grid.best_params_,
    "best_score": mlp_grid.best_score_
})

print("MLP tuning complete.")

# ------------------------------------------------------------
# 2. RBF SVM tuning
# ------------------------------------------------------------
svm_param_grid = {
    "C": [1, 10],
    "gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    SVC(kernel="rbf", probability=True, random_state=42),
    svm_param_grid,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=-1
)

svm_grid.fit(X_train_df, y_train)

tuned_results.append({
    "classifier": "RBF SVM (tuned)",
    "best_params": svm_grid.best_params_,
    "best_score": svm_grid.best_score_
})

print("SVM tuning complete.")

# ------------------------------------------------------------
# 3. Random Forest tuning
# ------------------------------------------------------------
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 20]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=cv_strategy,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train_df, y_train)

tuned_results.append({
    "classifier": "Random Forest (tuned)",
    "best_params": rf_grid.best_params_,
    "best_score": rf_grid.best_score_
})

print("Random Forest tuning complete.")

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------
tuned_df = pd.DataFrame(tuned_results)
tuned_df = tuned_df.sort_values(by="best_score", ascending=False)

print("\nTuned classifier results:\n")
print(tuned_df)


Starting light hyperparameter tuning...

MLP tuning complete.
SVM tuning complete.
Random Forest tuning complete.

Tuned classifier results:

              classifier                                        best_params  best_score
0            MLP (tuned)  {'alpha': 0.001, 'hidden_layer_sizes': (128, 64)}    0.940581
1        RBF SVM (tuned)                         {'C': 10, 'gamma': 'auto'}    0.939260
2  Random Forest (tuned)             {'max_depth': 20, 'n_estimators': 200}    0.923717


In [ ]:
# ============================================================
# Cell 8 — Save Classifier Comparison Results
# ============================================================

# Save baseline comparison results
BASELINE_RESULTS_PATH = os.path.join(BASE_DIR, "classifier_comparison_baseline.csv")
results_df.to_csv(BASELINE_RESULTS_PATH, index=False)

# Save tuned results
TUNED_RESULTS_PATH = os.path.join(BASE_DIR, "classifier_comparison_tuned.csv")
tuned_df.to_csv(TUNED_RESULTS_PATH, index=False)

print("Results saved successfully.\n")

print("Baseline results saved to:")
print(BASELINE_RESULTS_PATH)

print("\nTuned results saved to:")
print(TUNED_RESULTS_PATH)

# ------------------------------------------------------------
# Create compact table for reporting (Top 5)
# ------------------------------------------------------------

report_table = results_df.head(5)[
    ["classifier", "accuracy", "f1_score", "roc_auc"]
].copy()

report_table = report_table.round(4)

print("\nTop 5 Classifiers (Baseline):\n")
print(report_table)


Results saved successfully.

Baseline results saved to:
/content/drive/MyDrive/DIP_Project/classifier_comparison_baseline.csv

Tuned results saved to:
/content/drive/MyDrive/DIP_Project/classifier_comparison_tuned.csv

Top 5 Classifiers (Baseline):

          classifier  accuracy  f1_score  roc_auc
0      MLPClassifier    0.8646    0.8646   0.9357
1            RBF SVM    0.8464    0.8464   0.9258
2      Random Forest    0.8443    0.8443   0.9218
3        Extra Trees    0.8393    0.8393   0.9199
4  Gradient Boosting    0.8099    0.8098   0.8977


In [ ]:
# ============================================================
# Cell 9 — Final Classifier Selection Summary
# ============================================================

print("Final Classifier Selection Summary\n")

print("Top Tuned Classifiers:")
print(tuned_df)

best_model = tuned_df.iloc[0]

print("\nSelected Final Classifier:")
print(f"Classifier : {best_model['classifier']}")
print(f"Best Params: {best_model['best_params']}")
print(f"ROC-AUC    : {best_model['best_score']:.6f}")

print("\nKey Observations:")
print("- MLPClassifier achieved the highest performance after tuning.")
print("- RBF SVM performed nearly as well, indicating strong feature separability.")
print("- Random Forest also showed strong performance as a non-neural baseline.")
print("- Results suggest DIP features provide strong discriminative power independent of classifier type.")


Final Classifier Selection Summary

Top Tuned Classifiers:
              classifier                                        best_params  best_score
0            MLP (tuned)  {'alpha': 0.001, 'hidden_layer_sizes': (128, 64)}    0.940581
1        RBF SVM (tuned)                         {'C': 10, 'gamma': 'auto'}    0.939260
2  Random Forest (tuned)             {'max_depth': 20, 'n_estimators': 200}    0.923717

Selected Final Classifier:
Classifier : MLP (tuned)
Best Params: {'alpha': 0.001, 'hidden_layer_sizes': (128, 64)}
ROC-AUC    : 0.940581

Key Observations:
- MLPClassifier achieved the highest performance after tuning.
- RBF SVM performed nearly as well, indicating strong feature separability.
- Random Forest also showed strong performance as a non-neural baseline.
- Results suggest DIP features provide strong discriminative power independent of classifier type.
